### Imports

In [4]:
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from Springer_Segmentation.gerar_modelo import treinar_modelo_springer
from source.extrair_features import extrair_features

# Segmentação de Springer
O algoritmo proposto por Springer et al. (2016) é o estado da arte para a segmentação de sinais de fonocardiograma (PCG). Utilizando Modelos Ocultos de Markov com Duração Explícita (HSMM) e o algoritmo de Viterbi, o modelo analisa características do áudio (como Transformadas de Wavelet e Energia de Shannon) para delinear o ciclo cardíaco com precisão fisiológica.

Sua função neste projeto é receber o áudio bruto e classificar o sinal em quatro fases mecânicas fundamentais:

- S1 (Primeira Bulha): Início da sístole (fechamento das válvulas mitral e tricúspide).

- Sístole: Período de ejeção ventricular.

- S2 (Segunda Bulha): Início da diástole (fechamento das válvulas aórtica e pulmonar).

- Diástole: Período de relaxamento e enchimento.

A partir dessa marcação temporal confiável, torna-se possível extrair as métricas de duração e amplitude que alimentarão os classificadores de Machine Learning nas etapas seguintes.

## Gerando o Modelo de Segmentação de Springer

In [ ]:
# Configure os caminhos do seu computador
CAMINHO_TCC = r"C:\Users\gusta\TCC\TCC"
PASTA_REPO = os.path.join(CAMINHO_TCC, "Springer_Segmentation")

# A pasta de treino do Physionet e onde o modelo vai nascer
PASTA_PHYSIONET = os.path.join(PASTA_REPO, "training_data")
MODELO_PKL = os.path.join(CAMINHO_TCC, "springer_segmentation_model.pkl")

modelo_final = treinar_modelo_springer(
    caminho_repo = PASTA_REPO,
    pasta_dados = PASTA_PHYSIONET,
    arquivo_saida = MODELO_PKL,
    max_audios = 1000
)

## Extraindo as features

In [ ]:
CAMINHO_TCC = r"C:\Users\gusta\TCC\TCC"
PASTA_REPO = os.path.join(CAMINHO_TCC, "Springer_Segmentation")
MODELO_PKL = r"springer_segmentation_model.pkl"
PASTA_AUDIOS = r"C:\Users\gusta\TCC\TCC\data\train"
ARQUIVO_SAIDA = os.path.join(CAMINHO_TCC, "features_extraidas.csv")

features_extraidas = extrair_features(
    caminho_repo = PASTA_REPO,
    caminho_modelo = MODELO_PKL,
    pasta_audios = PASTA_AUDIOS,
    arquivo_saida = ARQUIVO_SAIDA
)

features_extraidas.head()

### 🔗 1. Fusão de Dados (Data Fusion)
Nesta etapa, unificamos as características temporais extraídas dos áudios com os rótulos clínicos e metadados dos pacientes. Como o arquivo original de treino possui os áudios dispostos em colunas (formato *wide*), utilizamos a técnica de *Melt* (unpivot) para transformar cada gravação em uma linha individual (formato *long*), permitindo o cruzamento exato com as *features* matemáticas.

In [7]:
caminho_features = 'features_extraidas.csv'
caminho_train = r'data\train.csv'
caminho_meta = r'data\additional_metadata.csv'

# 1. Carregar as tabelas
df_features = pd.read_csv(caminho_features)
df_train = pd.read_csv(caminho_train)
df_meta = pd.read_csv(caminho_meta)

# 2. Reorganizar o Train.csv (Melt) e preparar as chaves de cruzamento
colunas_audios = [f'recording_{i}' for i in range(1, 9)]
df_train_linhas = df_train.melt(
    id_vars=['patient_id', 'AS', 'AR', 'MR', 'MS', 'N'], 
    value_vars=colunas_audios, 
    value_name='nome_do_audio_sem_wav'
).dropna(subset=['nome_do_audio_sem_wav']) # Já remove os vazios na mesma linha

df_train_linhas['arquivo_wav'] = df_train_linhas['nome_do_audio_sem_wav'].astype(str) + '.wav'
df_features = df_features.rename(columns={'patient_id': 'arquivo_wav'})

# 3. A Grande Fusão (Merge)
df_rotulos_completos = pd.merge(df_train_linhas, df_meta, on='patient_id', how='left')
dataset_final = pd.merge(df_features, df_rotulos_completos, on='arquivo_wav', how='inner')

# Limpar colunas temporárias que não precisamos mais
dataset_final = dataset_final.drop(columns=['variable', 'nome_do_audio_sem_wav'])

# --- INÍCIO DAS TRANSFORMAÇÕES NUMÉRICAS (No mesmo DataFrame) ---

# 4. Encoding (Transformar em números)
dataset_final['Gender'] = dataset_final['Gender'].map({'M': 1, 'F': 0})
dataset_final['Lives'] = dataset_final['Lives'].map({'U': 1, 'R': 0})

# 5. Converter Idade de Texto para Número Contínuo
def converter_idade(idade_str):
    if pd.isna(idade_str): return np.nan
    try:
        partes = str(idade_str).split('-')
        return (int(partes[0]) + int(partes[1])) / 2.0
    except:
        return np.nan 

dataset_final['Age'] = dataset_final['Age'].apply(converter_idade)

# 6. Preencher valores vazios com a média
colunas_para_preencher = ['Age', 'Gender', 'Smoker', 'Lives']
for col in colunas_para_preencher:
    dataset_final[col] = dataset_final[col].fillna(dataset_final[col].mean())

# 7. Reorganizar as colunas para ficar elegante
colunas_organizadas = ['patient_id', 'arquivo_wav', 'AS', 'AR', 'MR', 'MS', 'N', 
                       'Age', 'Gender', 'Smoker', 'Lives'] + [col for col in df_features.columns if col != 'arquivo_wav']
dataset_final = dataset_final[colunas_organizadas]

dataset_final = dataset_final.dropna(how='any')

caminho_salvar = 'dataset_final.csv'
dataset_final.to_csv(caminho_salvar, index=False)

print(f"Dimensões finais: {dataset_final.shape}")

# Exibe as primeiras linhas direto no Notebook
dataset_final.head()

Dimensões finais: (864, 31)


,patient_id,arquivo_wav,AS,AR,MR,MS,N,Age,Gender,Smoker,...,m_Ratio_SysRR,sd_Ratio_SysRR,m_Ratio_DiaRR,sd_Ratio_DiaRR,m_Ratio_SysDia,sd_Ratio_SysDia,m_Amp_SysS1,sd_Amp_SysS1,m_Amp_DiaS2,sd_Amp_DiaS2
0,patient_016,AR_016_sit_Aor.wav,0.0,1.0,0.0,0.0,0.0,34.5,1,1,...,0.091438,0.019738,0.660700,0.034412,0.140255,0.039258,0.412363,0.123084,0.884418,0.351987
1,patient_016,AR_016_sit_Mit.wav,0.0,1.0,0.0,0.0,0.0,34.5,1,1,...,0.107745,0.023489,0.641216,0.037205,0.169973,0.042578,0.391649,0.132505,0.898236,0.316727
2,patient_016,AR_016_sit_Pul.wav,0.0,1.0,0.0,0.0,0.0,34.5,1,1,...,0.222610,0.007361,0.526831,0.016333,0.423049,0.021189,0.201668,0.049161,0.787470,0.258047
3,patient_016,AR_016_sit_Tri.wav,0.0,1.0,0.0,0.0,0.0,34.5,1,1,...,0.217510,0.014600,0.539695,0.022940,0.404594,0.042584,0.279554,0.205512,0.716258,0.249561
4,patient_016,AR_016_sup_Aor.wav,0.0,1.0,0.0,0.0,0.0,34.5,1,1,...,0.042312,0.008571,0.704802,0.053228,0.061668,0.022319,0.357486,0.267772,0.834465,0.384031


### Seleção de Modelos (Benchmarking)
O objetivo principal deste projeto é diagnosticar simultaneamente a presença de quatro patologias valvulares (AS, AR, MR, MS) e a ausência delas (N). Como um paciente pode apresentar múltiplas condições concomitantes, o problema é modelado como Classificação Multilabel.

Para algoritmos que não suportam saídas múltiplas nativamente, adotamos a estratégia `MultiOutputClassifier`, que ajusta um classificador independente para cada classe. A avaliação foca no F1-Score Macro e, em especial, no PhysioNet Score Macro (a média aritmética entre Sensibilidade e Especificidade adaptada para múltiplas classes), uma métrica clínica rigorosa que recompensa a segurança de detectar a doença e penaliza falsos alarmes.

Nesta etapa, treinamos e avaliamos múltiplos algoritmos de Machine Learning na sua configuração padrão (baseline). O objetivo é identificar qual arquitetura consegue capturar melhor os padrões das características extraídas (Wavelets e durações de Liu et al.) para este cenário de diagnósticos múltiplos.

Para garantir uma comparação justa entre modelos baseados em árvores e modelos lineares/distância, os dados preditores (X) foram normalizados utilizando o `StandardScaler`.

Modelos Testados:
1. Random Forest (Floresta Aleatória)
2. Gradient Boosting (Árvores em Cascata)
3. Regressão Logística (Estatística Clássica)
4. Support Vector Machine - SVM (Fronteiras de Decisão)
5. K-Nearest Neighbors - KNN (Agrupamento por Similaridade)

In [ ]:
# 1. Carregar os Dados
caminho_dataset = 'dataset_final.csv'
df = pd.read_csv(caminho_dataset)

# 2. Separar Preditores (X) e os Alvos (y)
# Agora o nosso Target são as 5 colunas de diagnóstico!
alvos = ['AS', 'AR', 'MR', 'MS', 'N']
colunas_para_remover = ['patient_id', 'arquivo_wav'] + alvos

X = df.drop(columns=colunas_para_remover)
y = df[alvos] 

# 3. Dividir em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Padronização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. O Grid de Largada (Adaptado para Multilabel)
# Random Forest e KNN suportam nativamente. Os demais ganham a "capa" MultiOutput.
modelos = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Gradient Boosting": MultiOutputClassifier(GradientBoostingClassifier(random_state=42)),
    "Regressão Logística": MultiOutputClassifier(LogisticRegression(max_iter=1000, random_state=42)),
    "SVM (Máquina de Vetores)": MultiOutputClassifier(SVC(random_state=42))
}

def score_physionet_macro(y_true, y_pred):
    """
    Adaptação do PhysioNet Challenge Score (2016) para o cenário Multilabel (2024).
    Calcula (Sensibilidade + Especificidade)/2 para cada doença separadamente e tira a média geral.
    """
    scores = []
    # Itera sobre cada uma das 5 colunas (doenças)
    for col in range(y_true.shape[1]):
        # Extrai os acertos e erros daquela doença específica
        tn, fp, fn, tp = confusion_matrix(y_true.iloc[:, col], y_pred[:, col]).ravel()
        
        # Calcula as duas métricas isoladas (evita divisão por zero)
        sensibilidade = tp / (tp + fn) if (tp + fn) > 0 else 0
        especificidade = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        # O Score do PhysioNet para esta doença
        score_doenca = (sensibilidade + especificidade) / 2
        scores.append(score_doenca)
        
    return np.mean(scores) # A média (Macro) de todas as doenças

# 6. Benchmarking
resultados = []

print("Treinando modelos...")
for nome_modelo, algoritmo in modelos.items():
    # Treino
    algoritmo.fit(X_train_scaled, y_train)
    
    # Previsão
    y_pred = algoritmo.predict(X_test_scaled)
    
    # Avaliação Multilabel
    acc_exata = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
    score_tcc = score_physionet_macro(y_test, y_pred)
    
    resultados.append({
        "Modelo": nome_modelo,
        "Acurácia Exata (%)": round(acc_exata * 100, 2),
        "F1-Score Macro (%)": round(f1_macro * 100, 2),
        "F1-Score Micro (%)": round(f1_micro * 100, 2),
        "Score PhysioNet Macro (%)": round(score_tcc * 100, 2)
    })

# 7. O Pódio
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values(by="Score PhysioNet Macro (%)", ascending=False).reset_index(drop=True)

# Cores roxas para diferenciar do teste binário
df_resultados.style.background_gradient(cmap='Purples')

Treinando modelos (Isso pode levar alguns segundos devido às múltiplas saídas)...


,Modelo,Acurácia Exata (%),F1-Score Macro (%),F1-Score Micro (%),Score PhysioNet Macro (%)
0,Random Forest,46.240000,72.360000,70.700000,78.710000
1,SVM (Máquina de Vetores),41.620000,70.500000,68.770000,77.590000
2,Gradient Boosting,38.730000,69.900000,67.990000,77.140000
3,KNN,46.240000,69.020000,67.660000,76.670000
4,Regressão Logística,36.990000,66.470000,64.580000,75.010000
